# Landlab Variables from GEE

This notebook reproduces the Google Earth Engine workflow in Python for notebook use.

It does the following:
- loads an AOI from a local shapefile
- computes temporal median products for MODIS LAI, ERA5 SWE, SMAP L4 surface soil moisture, MODIS snow cover, and Landsat 8 NDVI
- displays those layers on an interactive map
- prints AOI-level median values
- optionally exports the rasters to Google Drive


In [ ]:
# If needed:
# !pip install -q earthengine-api geemap geopandas


In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
from pathlib import Path


In [ ]:
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()


## Controls

Update the AOI path, date range, and export folder as needed.


In [ ]:
# ============================================================
# USER INPUTS
# ============================================================

AOI_PATH = "/mnt/c/Users/amehedi/Downloads/eaglecreek/eaglecreek.shp"
START_DATE = "2024-06-08"
END_DATE = "2025-01-01"

DRIVE_FOLDER = "GEE_exports"

SCALE_LAI = 500
SCALE_SWE = 11132
SCALE_SMAP = 9000
SCALE_SNOW = 500
SCALE_NDVI = 30


## Load AOI

The AOI is read from a local shapefile, reprojected to WGS84, and converted to an Earth Engine geometry.


In [ ]:
aoi_gdf = gpd.read_file(AOI_PATH)

if aoi_gdf.crs is None:
    raise ValueError("AOI shapefile has no CRS.")

aoi_gdf = aoi_gdf.to_crs("EPSG:4326")
aoi_union = aoi_gdf.geometry.union_all()
aoi = ee.Geometry(aoi_union.__geo_interface__)

aoi_gdf.head()


In [ ]:
# ============================================================
# HELPER: export image to Google Drive
# ============================================================

def export_image_to_drive(img, name, scale, folder=DRIVE_FOLDER, region=None):
    if region is None:
        region = aoi

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=name,
        folder=folder,
        fileNamePrefix=name,
        region=region,
        scale=scale,
        maxPixels=1e13,
        fileFormat="GeoTIFF",
    )
    task.start()
    return task


## MODIS LAI


In [ ]:
MODIS_LAI = (
    ee.ImageCollection("MODIS/061/MCD15A3H")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(aoi)
)

def prep_lai(img):
    qc = img.select("FparLai_QC")
    good_qc = qc.bitwiseAnd(3).eq(0)

    lai = img.select("Lai").multiply(0.1).rename("LAI")

    return (
        lai.updateMask(good_qc)
        .clip(aoi)
        .copyProperties(img, ["system:time_start"])
    )

lai_median = MODIS_LAI.map(prep_lai).median().clip(aoi)

lai_aoi_median = lai_median.reduceRegion(
    reducer=ee.Reducer.median(),
    geometry=aoi,
    scale=SCALE_LAI,
    maxPixels=1e13,
)

print("AOI median LAI:", lai_aoi_median.getInfo())


## ERA5 SWE


In [ ]:
ERA5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(aoi)
    .select("snow_depth_water_equivalent")
)

swe_median = ERA5.median().clip(aoi).rename("SWE")

swe_aoi_median = swe_median.reduceRegion(
    reducer=ee.Reducer.median(),
    geometry=aoi,
    scale=SCALE_SWE,
    maxPixels=1e13,
)

print("AOI median SWE:", swe_aoi_median.getInfo())


## SMAP L4 Surface Soil Moisture


In [ ]:
SMAPL4 = (
    ee.ImageCollection("NASA/SMAP/SPL4SMGP/008")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(aoi)
    .select("sm_surface")
)

sm_median = SMAPL4.median().clip(aoi).rename("SM_surface")

sm_aoi_median = sm_median.reduceRegion(
    reducer=ee.Reducer.median(),
    geometry=aoi,
    scale=SCALE_SMAP,
    maxPixels=1e13,
)

print("AOI median SMAP soil moisture:", sm_aoi_median.getInfo())


## MODIS Snow Cover


In [ ]:
MODIS_SNOW = (
    ee.ImageCollection("MODIS/061/MOD10A2")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(aoi)
)

def prep_snow(img):
    sca = img.select("Maximum_Snow_Extent")
    valid = sca.gte(0).And(sca.lte(100))

    return (
        sca.updateMask(valid)
        .clip(aoi)
        .rename("SnowCover")
        .copyProperties(img, ["system:time_start"])
    )

snow_median = MODIS_SNOW.map(prep_snow).median().clip(aoi)

snow_aoi_median = snow_median.reduceRegion(
    reducer=ee.Reducer.median(),
    geometry=aoi,
    scale=SCALE_SNOW,
    maxPixels=1e13,
)

print("AOI median snow cover:", snow_aoi_median.getInfo())


## Landsat 8 NDVI


In [ ]:
L8 = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(aoi)
)

def prep_ndvi(img):
    qa = img.select("QA_PIXEL")

    clear_mask = (
        qa.bitwiseAnd(1 << 3).eq(0)
        .And(qa.bitwiseAnd(1 << 4).eq(0))
    )

    red = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B5").multiply(0.0000275).add(-0.2)

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")

    return (
        ndvi.updateMask(clear_mask)
        .clip(aoi)
        .copyProperties(img, ["system:time_start"])
    )

ndvi_median = L8.map(prep_ndvi).median().clip(aoi)

ndvi_aoi_median = ndvi_median.reduceRegion(
    reducer=ee.Reducer.median(),
    geometry=aoi,
    scale=SCALE_NDVI,
    maxPixels=1e13,
)

print("AOI median NDVI:", ndvi_aoi_median.getInfo())


## Interactive Map


In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, 9)

Map.addLayer(lai_median,  {"min": 0, "max": 8,   "palette": ["yellow", "green", "darkgreen"]}, "Median LAI")
Map.addLayer(swe_median,  {"min": 0, "max": 0.5, "palette": ["white", "lightblue", "blue"]}, "Median SWE")
Map.addLayer(sm_median,   {"min": 0, "max": 0.6, "palette": ["brown", "yellow", "cyan", "blue"]}, "Median SMAP Surface SM")
Map.addLayer(snow_median, {"min": 0, "max": 100, "palette": ["black", "gray", "white"]}, "Median Snow Cover")
Map.addLayer(ndvi_median, {"min": -0.2, "max": 0.9, "palette": ["brown", "yellow", "green"]}, "Median NDVI")

Map.addLayer(aoi, {"color": "red"}, "AOI")
Map


## Export to Google Drive


In [ ]:
tasks = {}

tasks["lai"] = export_image_to_drive(
    lai_median,
    f"Median_LAI_{START_DATE}_{END_DATE}",
    SCALE_LAI,
)

tasks["swe"] = export_image_to_drive(
    swe_median,
    f"Median_SWE_{START_DATE}_{END_DATE}",
    SCALE_SWE,
)

tasks["smap"] = export_image_to_drive(
    sm_median,
    f"Median_SMAP_SM_{START_DATE}_{END_DATE}",
    SCALE_SMAP,
)

tasks["snow"] = export_image_to_drive(
    snow_median,
    f"Median_SnowCover_{START_DATE}_{END_DATE}",
    SCALE_SNOW,
)

tasks["ndvi"] = export_image_to_drive(
    ndvi_median,
    f"Median_NDVI_{START_DATE}_{END_DATE}",
    SCALE_NDVI,
)

print("Started export tasks:")
for k, v in tasks.items():
    print(k, v.status()["state"])


## Check Task Status


In [ ]:
for k, v in tasks.items():
    print(k, v.status())


## AOI Median Summary Table


In [ ]:
summary = pd.DataFrame(
    [
        {"variable": "LAI", "median": lai_aoi_median.getInfo().get("LAI")},
        {"variable": "SWE", "median": swe_aoi_median.getInfo().get("SWE")},
        {"variable": "SM_surface", "median": sm_aoi_median.getInfo().get("SM_surface")},
        {"variable": "SnowCover", "median": snow_aoi_median.getInfo().get("SnowCover")},
        {"variable": "NDVI", "median": ndvi_aoi_median.getInfo().get("NDVI")},
    ]
)

summary